## Part 1

In [ ]:
import pandas as pd

data_url = "https://d396qusza40orc.cloudfront.net/exdata%2Fdata%2Fhousehold_power_consumption.zip"
try:
    df_energy = pd.read_csv(
        data_url,
        sep=';',
        na_values=['?']
    )
    print("ok")

except Exception as e:
    print(f"An error occurred while loading the data: {e}")
    df_energy = None

ok


In [ ]:
if df_energy is not None:
    df_energy['Datetime'] = pd.to_datetime(
        df_energy['Date'] + ' ' + df_energy['Time'],
        format='%d/%m/%Y %H:%M:%S'
    )
    df_energy = df_energy.set_index('Datetime')
    df_energy = df_energy.drop(['Date', 'Time'], axis=1)

    print("First two rows:")
    print(df_energy.head(2))
else:
    print("An error occurred")


First two rows:
                     Global_active_power  Global_reactive_power  Voltage  \
Datetime                                                                   
2006-12-16 17:24:00                4.216                  0.418   234.84   
2006-12-16 17:25:00                5.360                  0.436   233.63   

                     Global_intensity  Sub_metering_1  Sub_metering_2  \
Datetime                                                                
2006-12-16 17:24:00              18.4             0.0             1.0   
2006-12-16 17:25:00              23.0             0.0             1.0   

                     Sub_metering_3  
Datetime                             
2006-12-16 17:24:00            17.0  
2006-12-16 17:25:00            16.0  


In [ ]:
if df_energy is not None:

    numeric_columns = df_energy.columns.tolist()
    df_energy[numeric_columns] = df_energy[numeric_columns].apply(pd.to_numeric)
    print(df_energy.dtypes)
else:
    print("An error occurred")

Global_active_power      float64
Global_reactive_power    float64
Voltage                  float64
Global_intensity         float64
Sub_metering_1           float64
Sub_metering_2           float64
Sub_metering_3           float64
dtype: object


In [ ]:
import requests
weather_url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    'latitude': 48.85,
    'longitude': 2.35,
    'start_date': '2006-12-16',
    'end_date': '2010-11-26',
    'hourly': ['temperature_2m', 'precipitation', 'wind_speed_10m'],
    'timezone': 'Europe/Berlin'
}
df_weather = None
try:
    response = requests.get(weather_url, params=params)
    response.raise_for_status()
    weather_data = response.json()

    df_weather = pd.DataFrame(weather_data['hourly'])

    df_weather['time'] = pd.to_datetime(df_weather['time'])
    df_weather = df_weather.set_index('time')
    df_weather.index.name = 'Datetime'
    print(len(df_weather))
except requests.exceptions.RequestException as e:
    print(e)
    print('An error occurred')
    df_weather = None

34608


In [ ]:
if df_energy is not None and df_weather is not None:
    df_combined = df_energy.join(df_weather)
    print(len(df_combined))
    print(df_combined.head(3))
else:
    print("An error occurred")
    df_combined = 0

2075259
                     Global_active_power  Global_reactive_power  Voltage  \
Datetime                                                                   
2006-12-16 17:24:00                4.216                  0.418   234.84   
2006-12-16 17:25:00                5.360                  0.436   233.63   
2006-12-16 17:26:00                5.374                  0.498   233.29   

                     Global_intensity  Sub_metering_1  Sub_metering_2  \
Datetime                                                                
2006-12-16 17:24:00              18.4             0.0             1.0   
2006-12-16 17:25:00              23.0             0.0             1.0   
2006-12-16 17:26:00              23.0             0.0             2.0   

                     Sub_metering_3  temperature_2m  precipitation  \
Datetime                                                             
2006-12-16 17:24:00            17.0             NaN            NaN   
2006-12-16 17:25:00            16.0

## Part 2

In [ ]:
if df_combined is not None:
    len_before = len(df_combined)
    df_clean = df_combined.dropna()
    len_after = len(df_clean)

    aggregation = {
        'Global_active_power': 'sum',
        'Global_reactive_power': 'sum',
        'Voltage': 'mean',
        'Global_intensity': 'mean',
        'Sub_metering_1': 'sum',
        'Sub_metering_2': 'sum',
        'Sub_metering_3': 'sum',
        'temperature_2m': 'mean',
        'precipitation': 'sum',
        'wind_speed_10m': 'mean'
    }

    df_daily = df_clean.resample('D').agg(aggregation)
    df_daily = df_daily.dropna()

    print(len(df_daily))
    print(df_daily.head(3))
else:
    print("An error occurred")
    df_daily = None

1433
            Global_active_power  Global_reactive_power     Voltage  \
Datetime                                                             
2006-12-16               19.254                  0.656  235.605000   
2006-12-17               55.036                  4.284  239.771250   
2006-12-18               33.404                  2.582  241.310417   

            Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  \
Datetime                                                                       
2006-12-16              13.6             0.0             2.0            85.0   
2006-12-17               9.8             3.0            55.0           228.0   
2006-12-18               5.8            38.0            45.0           194.0   

            temperature_2m  precipitation  wind_speed_10m  
Datetime                                                   
2006-12-16        5.316667            0.2        8.416667  
2006-12-17        3.050000            0.0        7.945833  
200

## Part 3

### Loading libraries

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import PolynomialFeatures
import joblib

from contextlib import asynccontextmanager
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

### Preparing training and test datasets

In [ ]:
def split_train_test(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """
    Splits the data according to the standard scenario: 80% training, 20% test

    Args:
        df(pd.DataFrame): DataFrame with combined energy consumption and meteorological conditions data
    Returns:
        tuple: Training and test sets for dependent and independent variables
    """
    # Forecasted consumption will be the dependent variable
    y = df['Global_active_power']

    independent_variables = ['temperature_2m','precipitation','wind_speed_10m','Global_reactive_power','Voltage','Global_intensity','Sub_metering_1','Sub_metering_2','Sub_metering_3']
    x = df[independent_variables]

    # Using shuffle=False because we are working with time series
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, shuffle=False)

    return x_train, y_train, x_test, y_test

### Linear_regression

In [ ]:
def linear_regression(x_train: pd.DataFrame, y_train: pd.Series,
                      x_test: pd.DataFrame, y_test: pd.Series) -> tuple[float, float]:
    """
    Trains a linear regression model, makes predictions on the test set, evaluates the model and saves it to a file

    Args:
        x_train (pd.DataFrame): Training feature set (independent variables)
        y_train (pd.Series): Training target values (energy consumption)
        x_test (pd.DataFrame): Test feature set
        y_test (pd.Series): Test target values

    Returns:
        tuple[float, float]: Tuple containing the mean squared error (MSE) and coefficient of determination (R2) for the model
    """
    # Train the model
    linear_model = LinearRegression()
    linear_model.fit(x_train, y_train)

    # Create predictions and evaluate
    y_prediction = linear_model.predict(x_test)
    mse = mean_squared_error(y_test, y_prediction)
    r2 = r2_score(y_test, y_prediction)

    # Save the model (for task 4)
    joblib.dump(linear_model, 'linear_model.pkl')

    return mse, r2

def polynomial_regression_degree_2(x_train: pd.DataFrame, y_train: pd.Series,
                                   x_test: pd.DataFrame, y_test: pd.Series) -> tuple[float, float]:
    """
    Trains a polynomial regression model, makes predictions on the test set, evaluates the model and saves it to a file

    Args:
        x_train (pd.DataFrame): Training feature set (independent variables)
        y_train (pd.Series): Training target values (energy consumption)
        x_test (pd.DataFrame): Test feature set
        y_test (pd.Series): Test target values

    Returns:
        tuple[float, float]: Tuple containing the mean squared error (MSE) and coefficient of determination (R2) for the model
    """
    # Avoid redundancy by creating a transformer - with include_bias=False, no column of ones is added
    polynomial_transformer = PolynomialFeatures(degree=2, include_bias=False)

    # Transform features to polynomial
    x_train_polynomial = polynomial_transformer.fit_transform(x_train)
    x_test_polynomial = polynomial_transformer.transform(x_test)

    # Train the model
    polynomial_model = LinearRegression()
    polynomial_model.fit(x_train_polynomial, y_train)

    # Create predictions and evaluate
    y_prediction = polynomial_model.predict(x_test_polynomial)
    mse = mean_squared_error(y_test, y_prediction)
    r2 = r2_score(y_test, y_prediction)

    # Save the model and transformer (for task 4)
    joblib.dump(polynomial_transformer, 'polynomial_transformer.pkl')
    joblib.dump(polynomial_model, 'polynomial_model.pkl')

    return mse, r2

def compare_results(linear_results: tuple[float, float], polynomial_results: tuple[float, float]) -> None:
    """
    Displays and compares evaluation metrics (MSE and R2) obtained from linear regression and polynomial regression

    Args:
        linear_results (tuple[float, float]): Tuple containing (MSE, R2) for the linear model
        polynomial_results (tuple[float, float]): Tuple containing (MSE, R2) for the polynomial model

    Returns:
        None: The function does not return a value, only prints
    """
    mse_lin, r2_lin = linear_results
    mse_poly, r2_poly = polynomial_results

    print(f"Linear Regression:\nMSE:{mse_lin:.3f}\nR2:{r2_lin:.3f}\n")
    print(f"Polynomial Regression:\nMSE:{mse_poly:.3f}\nR2:{r2_poly:.3f}")

In [ ]:
x_train, y_train, x_test, y_test = split_train_test(df_daily)
compare_results(linear_regression(x_train, y_train, x_test, y_test), polynomial_regression_degree_2(x_train, y_train, x_test, y_test))

Linear Regression:
MSE:4.980
R2:0.925

Polynomial Regression:
MSE:1.744
R2:0.974


Conclsuion: The results obtained for both models show that the polynomial regression model performs much better, as it has a significantly lower MSE error and also better fit, explaining nearly 5% more variance in the data.

## Part 4

In [ ]:
# Initialize models and transformer globally
linear_model, polynomial_model, polynomial_transformer = None, None, None

@asynccontextmanager
async def lifespan():
    """
    Manages the lifespan of the FastAPI application. Before yield, loads trained models and the transformer from .pkl files into global variables. After yield, shuts down the server.

    Returns:
        Generator: Asynchronous context generator
    """
    global linear_model, polynomial_model, polynomial_transformer

    try:
        linear_model = joblib.load('linear_model.pkl')
        polynomial_model = joblib.load('polynomial_model.pkl')
        polynomial_transformer = joblib.load('polynomial_transformer.pkl')
    except Exception as e:
        print(f"Error while loading files: {e}")
        linear_model, polynomial_model, polynomial_transformer = None, None, None
    yield

app = FastAPI(title="Daily energy consumption forecast for Sceaux", description="Using linear and 2nd degree polynomial regression", lifespan=lifespan)

class DailyFeatures(BaseModel):
    temperature_2m: float = Field(..., description="Average daily temperature (°C)")
    precipitation: float = Field(..., description="Total daily precipitation (mm)")
    wind_speed_10m: float = Field(..., description="Average daily wind speed (km/h)")
    global_reactive_power: float = Field(..., description="Daily sum of reactive power (kW)")
    voltage: float = Field(..., description="Average daily voltage (V)")
    global_intensity: float = Field(..., description="Average daily current intensity (A)")
    sub_metering_1: float = Field(..., description="Daily sum of sub-meter 1 consumption (Wh)")
    sub_metering_2: float = Field(..., description="Daily sum of sub-meter 2 consumption (Wh)")
    sub_metering_3: float = Field(..., description="Daily sum of sub-meter 3 consumption (Wh)")

# Define the endpoint
@app.post("/predict_daily_consumption", tags=["Forecasts"])
def predict_consumption(features: DailyFeatures) -> set:
    """
    Performs a daily electricity consumption forecast based on the submitted data

    Args:
        features (DailyFeatures): Object containing 9 input features (e.g. temperature, precipitation), validated by Pydantic

    Returns:
        set: Set containing forecasts in kilowatts (kW) from both models and a recommendation for which model is better

    Raises:
        HTTPException: Returns code 500 if models were not correctly loaded during server startup
    """
    if linear_model is None or polynomial_model is None:
         raise HTTPException(status_code=500, detail="Models were not loaded correctly")

    input_data = np.array([[
        features.temperature_2m, features.precipitation, features.wind_speed_10m,
        features.global_reactive_power, features.voltage, features.global_intensity,
        features.sub_metering_1, features.sub_metering_2, features.sub_metering_3
    ]])

    # Save model predictions
    pred_lin = linear_model.predict(input_data)[0]
    polynomial_data = polynomial_transformer.transform(input_data)
    pred_poly = polynomial_model.predict(polynomial_data)[0]

    return {
        f"Linear regression forecast (kW):{pred_lin:.3f}\n",
        f"Polynomial regression forecast (kW):{pred_poly:.3f}\n"
        "Recommended model is polynomial regression: better R2 and lower MSE"
    }

Conclusion: After launching uvicorn, the terminal shows that the server address is http://127.0.0.1:8000 (endpoint is /predict_daily_consumption), so you can already navigate through the terminal to view the API. However, due to the constraint of completing the entire project in a single notebook, I will allow myself to skip this step.